# Severstal OSR Full Experiment

This notebook runs the full `severstral-osr` pipeline:
1. export dataset stats and single-label lists
2. train stage-1 PatchCore once
3. run split pipelines (classifier + embeddings + OSR + cascade)
4. aggregate metrics

In [ ]:
# Optional compatibility reinstall (run only if your environment has package conflicts)
"""import sys, subprocess

# Reinstall a compatible scientific stack
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall",
    "numpy==1.26.4",
    "scipy==1.11.4",
    "scikit-learn==1.4.2",
    "pandas==2.2.2",
    "matplotlib==3.8.4",
    "pillow==10.3.0",
    "pyyaml==6.0.1",
    "tqdm==4.66.4",
])

print("Done. Now restart runtime: Runtime > Restart runtime")"""

In [8]:
"""!git clone https://github.com/spinelessknave8/FYP_code.git /content/FYP-code
%cd /content/FYP-code
!git fetch origin
!git reset --hard origin/main"""

fatal: destination path '/content/FYP-code' already exists and is not an empty directory.
/content
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.29 KiB | 658.00 KiB/s, done.
From https://github.com/spinelessknave8/FYP_code
   752fd69..4787698  main       -> origin/main
HEAD is now at 4787698 added more permutations for severstral only run


In [9]:
# Setup paths (Colab-friendly)
import os, sys, subprocess
from pathlib import Path

REPO = Path('/content/FYP-code')
if not REPO.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/spinelessknave8/FYP_code.git', str(REPO)])
else:
    subprocess.check_call(['git', '-C', str(REPO), 'pull', '--ff-only'])

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
if str(REPO / 'severstral-osr' / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'severstral-osr' / 'src'))

print('cwd:', Path.cwd())

cwd: /content/FYP-code


In [10]:
# Mount drive and generate colab configs
import yaml
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

sev = Path('/content/drive/MyDrive/datasets/severstal')
assert sev.exists(), f'Missing {sev}'
assert (sev / 'train.csv').exists(), 'Missing train.csv'
assert (sev / 'train_images').exists(), 'Missing train_images'

base = yaml.safe_load(Path('severstral-osr/configs/default.yaml').read_text())
base['device'] = 'cuda'
base['severstal']['data_root'] = str(sev)
base['output_dir'] = '/content/drive/MyDrive/fyp_outputs/severstral_osr'
base['export_dir'] = '/content/drive/MyDrive/fyp_outputs/severstral_osr/exports'

Path('severstral-osr/configs/default.colab.yaml').write_text(yaml.safe_dump(base, sort_keys=False))
print('wrote severstral-osr/configs/default.colab.yaml')

for s in ['a', 'b', 'c', 'd']:
    split_cfg = yaml.safe_load(Path(f'severstral-osr/configs/split_{s}.yaml').read_text())
    merged = yaml.safe_load(yaml.safe_dump(base))
    merged.update(split_cfg)
    out = Path(f'severstral-osr/configs/split_{s}.colab.yaml')
    out.write_text(yaml.safe_dump(merged, sort_keys=False))
    print('wrote', out)

Mounted at /content/drive
wrote severstral-osr/configs/default.colab.yaml
wrote severstral-osr/configs/split_a.colab.yaml
wrote severstral-osr/configs/split_b.colab.yaml
wrote severstral-osr/configs/split_c.colab.yaml
wrote severstral-osr/configs/split_d.colab.yaml


In [11]:
# Export Severstal defect stats and single-label lists
from export_single_label_lists import main as export_lists

export_lists('severstral-osr/configs/default.colab.yaml')

Severstal classes: ['Class_1', 'Class_2', 'Class_3', 'Class_4']
Single-label counts: {'Class_1': 769, 'Class_2': 195, 'Class_3': 4759, 'Class_4': 516}
Wrote exports to: /content/drive/MyDrive/fyp_outputs/severstral_osr/exports


In [ ]:
# Stage-1 PatchCore (run once)
from notebook_entrypoints import run_stage1

run_stage1('severstral-osr/configs/default.colab.yaml')

In [ ]:
# Run split pipelines (step-level checkpointing enabled)
import time
from notebook_entrypoints import run_split_pipeline

cfgs = [
    'severstral-osr/configs/split_a.colab.yaml',
    'severstral-osr/configs/split_b.colab.yaml',
    'severstral-osr/configs/split_c.colab.yaml',
    'severstral-osr/configs/split_d.colab.yaml',
]

for c in cfgs:
    t = time.time()
    run_split_pipeline(c, skip_if_complete=True)
    print('[done]', c, f'{time.time()-t:.1f}s')


In [ ]:
# Aggregate and print summary
import json
import pandas as pd
import yaml
from pathlib import Path

out_root = Path('/content/drive/MyDrive/fyp_outputs/severstral_osr')
rows = []
for s in ['split_a', 'split_b', 'split_c', 'split_d']:
    p_osr = out_root / s / 'osr' / 'metrics.json'
    p_cas = out_root / s / 'cascade' / 'metrics.json'
    if not p_osr.exists() or not p_cas.exists():
        print('missing metrics for', s)
        continue
    osr = json.loads(p_osr.read_text())
    cas = json.loads(p_cas.read_text())
    split_cfg_path = Path(f'severstral-osr/configs/{s}.yaml')
    known_classes = ''
    if split_cfg_path.exists():
        known_classes = ','.join(yaml.safe_load(split_cfg_path.read_text())['known_classes'])
    rows.append({
        'split': s,
        'known_classes': known_classes,
        'auroc_known_unknown': osr.get('auroc_known_unknown'),
        'tpr_unknown': osr.get('tpr_unknown'),
        'fpr_known': osr.get('fpr_known'),
        'open_set_acc': osr.get('open_set_acc'),
        'known_accuracy': osr.get('known_accuracy'),
        'tau_source': cas.get('patchcore_threshold_source'),
        'tau': cas.get('patchcore_threshold'),
        'kappa': cas.get('kappa'),
        'fusion_rule': cas.get('fusion_rule'),
        'stage1_pass_rate_known': cas.get('stage1_pass_rate_known'),
        'stage1_pass_rate_unknown': cas.get('stage1_pass_rate_unknown'),
        'tpr_unknown_system': cas.get('tpr_unknown_system'),
        'fpr_known_system': cas.get('fpr_known_system'),
    })

df = pd.DataFrame(rows).sort_values('split')
if len(df) > 0:
    display(df)
    display(df.mean(numeric_only=True).to_frame('mean').T)
    out_csv = out_root / 'severstral_osr_summary.csv'
    df.to_csv(out_csv, index=False)
    print('Wrote:', out_csv)
else:
    print('No metrics found yet.')


In [12]:
# Grid setup (cached-array sweep; no heavy PatchCore recompute)
import os, sys, json, yaml, numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score

REPO = Path('/content/FYP-code')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
if str(REPO / 'severstral-osr' / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'severstral-osr' / 'src'))

OUT = Path('/content/drive/MyDrive/fyp_outputs/severstral_osr')
SPLITS = ['split_a', 'split_b', 'split_c', 'split_d']

# Big grid
TAU_ACCEPT_RATES = [0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90,0.95]
KAPPAS = [0.10,0.20,0.30,0.40,0.50,0.60,0.70,0.80,0.90]
FUSIONS = ['and', 'or']
FPR_CAP = 0.10  # target system-level known false positive cap

print('OUT:', OUT)
print('splits:', SPLITS)
print('grid size per split:', len(TAU_ACCEPT_RATES)*len(KAPPAS)*len(FUSIONS))

OUT: /content/drive/MyDrive/fyp_outputs/severstral_osr
splits: ['split_a', 'split_b', 'split_c', 'split_d']
grid size per split: 252


In [13]:
# Run full cached sweep
rows = []
for split in SPLITS:
    cdir = OUT / split / 'cascade'
    req = [
        cdir / 'known_scores.npy',
        cdir / 'unknown_scores.npy',
        cdir / 'known_conf.npy',
        cdir / 'unknown_conf.npy',
        cdir / 'stage1_known_val_scores.npy',
    ]
    missing = [str(x) for x in req if not x.exists()]
    if missing:
        print('missing cached files for', split)
        for m in missing:
            print('  -', m)
        continue

    ks = np.load(cdir / 'known_scores.npy')
    us = np.load(cdir / 'unknown_scores.npy')
    kc = np.load(cdir / 'known_conf.npy')
    uc = np.load(cdir / 'unknown_conf.npy')
    kv = np.load(cdir / 'stage1_known_val_scores.npy')

    y = np.r_[np.zeros(len(ks)), np.ones(len(us))]
    s = np.r_[ks, us]
    auroc_score = float(roc_auc_score(y, s)) if len(np.unique(y)) > 1 else np.nan

    for tar in TAU_ACCEPT_RATES:
        tau = float(np.quantile(kv, tar))
        k_pass = ks > tau
        u_pass = us > tau

        for kappa in KAPPAS:
            for fr in FUSIONS:
                if fr == 'and':
                    k_rej = k_pass & (kc < kappa)
                    u_rej = u_pass & (uc < kappa)
                else:
                    k_rej = k_pass | (kc < kappa)
                    u_rej = u_pass | (uc < kappa)

                tpr_sys = float(np.mean(u_rej))
                fpr_sys = float(np.mean(k_rej))
                pass_k = float(np.mean(k_pass))
                pass_u = float(np.mean(u_pass))
                tpr_cond = float(np.mean(u_rej[u_pass])) if np.any(u_pass) else np.nan
                fpr_cond = float(np.mean(k_rej[k_pass])) if np.any(k_pass) else np.nan

                rows.append({
                    'split': split,
                    'tau_accept_rate': tar,
                    'tau': tau,
                    'kappa': kappa,
                    'fusion_rule': fr,
                    'auroc_score_only': auroc_score,
                    'stage1_pass_rate_known': pass_k,
                    'stage1_pass_rate_unknown': pass_u,
                    'gap_pass_unknown_minus_known': pass_u - pass_k,
                    'tpr_unknown_system': tpr_sys,
                    'fpr_known_system': fpr_sys,
                    'tpr_unknown_conditional': tpr_cond,
                    'fpr_known_conditional': fpr_cond,
                })

df_grid = pd.DataFrame(rows)
sweep_dir = OUT / 'sweeps' / 'severstral_grid'
sweep_dir.mkdir(parents=True, exist_ok=True)
grid_csv = sweep_dir / 'severstral_grid.csv'
df_grid.to_csv(grid_csv, index=False)
print('Wrote:', grid_csv, 'rows:', len(df_grid))
display(df_grid.head())

Wrote: /content/drive/MyDrive/fyp_outputs/severstral_osr/sweeps/severstral_grid/severstral_grid.csv rows: 1008


,split,tau_accept_rate,tau,kappa,fusion_rule,auroc_score_only,stage1_pass_rate_known,stage1_pass_rate_unknown,gap_pass_unknown_minus_known,tpr_unknown_system,fpr_known_system,tpr_unknown_conditional,fpr_known_conditional
0,split_a,0.3,2.794895,0.1,and,0.730859,0.70151,0.918605,0.217095,0.000000,0.00000,0.0,0.0
1,split_a,0.3,2.794895,0.1,or,0.730859,0.70151,0.918605,0.217095,0.918605,0.70151,1.0,1.0
2,split_a,0.3,2.794895,0.2,and,0.730859,0.70151,0.918605,0.217095,0.000000,0.00000,0.0,0.0
3,split_a,0.3,2.794895,0.2,or,0.730859,0.70151,0.918605,0.217095,0.918605,0.70151,1.0,1.0
4,split_a,0.3,2.794895,0.3,and,0.730859,0.70151,0.918605,0.217095,0.000000,0.00000,0.0,0.0


In [14]:
# Pick best GLOBAL config under FPR cap
g = (df_grid.groupby(['fusion_rule', 'tau_accept_rate', 'kappa'], as_index=False)
       .agg(
           tpr_unknown_system=('tpr_unknown_system', 'mean'),
           fpr_known_system=('fpr_known_system', 'mean'),
           pass_known=('stage1_pass_rate_known', 'mean'),
           pass_unknown=('stage1_pass_rate_unknown', 'mean'),
           gap=('gap_pass_unknown_minus_known', 'mean'),
       ))

feasible = g[g['fpr_known_system'] <= FPR_CAP].copy()
if len(feasible) == 0:
    print('No config meets FPR cap. Falling back to full set.')
    feasible = g.copy()

feasible = feasible.sort_values(
    ['tpr_unknown_system', 'fpr_known_system', 'gap'],
    ascending=[False, True, False],
)

display(feasible.head(20))
best = feasible.iloc[0].to_dict()
print('BEST GLOBAL:', best)

,fusion_rule,tau_accept_rate,kappa,tpr_unknown_system,fpr_known_system,pass_known,pass_unknown,gap
35,and,0.45,0.9,0.150941,0.094999,0.557632,0.551744,-0.005888
7,and,0.30,0.8,0.139909,0.069290,0.697850,0.683907,-0.013943
44,and,0.50,0.9,0.130708,0.087238,0.510471,0.505511,-0.004961
248,or,0.95,0.6,0.130577,0.084306,0.050701,0.056751,0.006050
16,and,0.35,0.8,0.127825,0.064997,0.649434,0.641295,-0.008139
25,and,0.40,0.8,0.117825,0.063274,0.607337,0.600113,-0.007224
238,or,0.90,0.5,0.113015,0.091658,0.086709,0.108853,0.022144
53,and,0.55,0.9,0.111640,0.075006,0.447235,0.466413,0.019178
237,or,0.90,0.4,0.109168,0.087290,0.086709,0.108853,0.022144
234,or,0.90,0.1,0.108853,0.086709,0.086709,0.108853,0.022144


BEST GLOBAL: {'fusion_rule': 'and', 'tau_accept_rate': 0.45, 'kappa': 0.9, 'tpr_unknown_system': 0.15094089650293593, 'fpr_known_system': 0.0949994334044075, 'pass_known': 0.5576318821504549, 'pass_unknown': 0.5517435678663483, 'gap': -0.005888314284106563}


In [15]:
# Apply best config to split colab YAMLs + OSR thresholds.json
best_fusion = str(best['fusion_rule'])
best_tau_accept = float(best['tau_accept_rate'])
best_kappa = float(best['kappa'])

print('Applying:', {
    'fusion_rule': best_fusion,
    'tau_accept_rate': best_tau_accept,
    'kappa': best_kappa,
})

for s in ['a','b','c','d']:
    cfgp = Path(f'severstral-osr/configs/split_{s}.colab.yaml')
    cfg = yaml.safe_load(cfgp.read_text())
    cfg.setdefault('two_stage', {})
    cfg['two_stage']['tau_source'] = 'target_known_val'
    cfg['two_stage']['tau_accept_rate'] = best_tau_accept
    cfg['two_stage']['fusion_rule'] = best_fusion
    cfgp.write_text(yaml.safe_dump(cfg, sort_keys=False))

    split = f'split_{s}'
    osr_dir = OUT / split / 'osr'
    osr_dir.mkdir(parents=True, exist_ok=True)
    thrp = osr_dir / 'thresholds.json'

    payload = {'tau': None, 'kappa': best_kappa}
    if thrp.exists():
        old = json.loads(thrp.read_text())
        payload['tau'] = old.get('tau')
    thrp.write_text(json.dumps(payload, indent=2))

    print('updated', cfgp, 'and', thrp)

Applying: {'fusion_rule': 'and', 'tau_accept_rate': 0.45, 'kappa': 0.9}
updated severstral-osr/configs/split_a.colab.yaml and /content/drive/MyDrive/fyp_outputs/severstral_osr/split_a/osr/thresholds.json
updated severstral-osr/configs/split_b.colab.yaml and /content/drive/MyDrive/fyp_outputs/severstral_osr/split_b/osr/thresholds.json
updated severstral-osr/configs/split_c.colab.yaml and /content/drive/MyDrive/fyp_outputs/severstral_osr/split_c/osr/thresholds.json
updated severstral-osr/configs/split_d.colab.yaml and /content/drive/MyDrive/fyp_outputs/severstral_osr/split_d/osr/thresholds.json


In [18]:
# Re-run cascade only with best config (fast path; reuses cached arrays)
from run_cascade import main as run_cascade_main
import time

for s in ['a','b','c','d']:
    cfg = f'severstral-osr/configs/split_{s}.colab.yaml'
    t0 = time.time()
    print('Running cascade:', cfg)
    run_cascade_main(cfg)
    print('done in', round(time.time() - t0, 1), 's')

Running cascade: severstral-osr/configs/split_a.colab.yaml
[severstal-cascade:split_a] loading cached known-val scores
[severstal-cascade:split_a] loading cached known scores
[severstal-cascade:split_a] loading cached unknown scores
[severstal-cascade:split_a] loading cached known confidences
[severstal-cascade:split_a] loading cached unknown confidences
[severstal-cascade:split_a] done in 7.0s
  stage1_pass_rate_known=0.5633
  stage1_pass_rate_unknown=0.8391
  tpr_unknown_system=0.0465 fpr_known_system=0.0778
done in 7.0 s
Running cascade: severstral-osr/configs/split_b.colab.yaml
[severstal-cascade:split_b] loading cached known-val scores
[severstal-cascade:split_b] loading cached known scores
[severstal-cascade:split_b] loading cached unknown scores
[severstal-cascade:split_b] loading cached known confidences
[severstal-cascade:split_b] loading cached unknown confidences
[severstal-cascade:split_b] done in 7.8s
  stage1_pass_rate_known=0.5714
  stage1_pass_rate_unknown=0.5913
  tpr_

In [19]:
# Final summary after best-grid config
rows = []
for split in SPLITS:
    p_osr = OUT / split / 'osr' / 'metrics.json'
    p_cas = OUT / split / 'cascade' / 'metrics.json'
    if not p_osr.exists() or not p_cas.exists():
        continue
    osr = json.loads(p_osr.read_text())
    cas = json.loads(p_cas.read_text())
    rows.append({
        'split': split,
        'auroc_known_unknown': osr.get('auroc_known_unknown'),
        'tpr_unknown': osr.get('tpr_unknown'),
        'fpr_known': osr.get('fpr_known'),
        'open_set_acc': osr.get('open_set_acc'),
        'known_accuracy': osr.get('known_accuracy'),
        'tau': cas.get('patchcore_threshold'),
        'kappa': cas.get('kappa'),
        'fusion_rule': cas.get('fusion_rule'),
        'stage1_pass_rate_known': cas.get('stage1_pass_rate_known'),
        'stage1_pass_rate_unknown': cas.get('stage1_pass_rate_unknown'),
        'tpr_unknown_system': cas.get('tpr_unknown_system'),
        'fpr_known_system': cas.get('fpr_known_system'),
    })

final_df = pd.DataFrame(rows).sort_values('split')
display(final_df)
display(final_df.mean(numeric_only=True).to_frame('mean').T)

out_csv = OUT / 'severstral_osr_summary_after_grid_best.csv'
final_df.to_csv(out_csv, index=False)
print('Wrote:', out_csv)

,split,auroc_known_unknown,tpr_unknown,fpr_known,open_set_acc,known_accuracy,tau,kappa,fusion_rule,stage1_pass_rate_known,stage1_pass_rate_unknown,tpr_unknown_system,fpr_known_system
0,split_a,0.611167,0.058140,0.095238,0.587509,0.939605,2.933017,0.9,and,0.563298,0.839147,0.046512,0.077816
1,split_b,0.588827,0.238495,0.138393,0.266506,0.825893,2.938982,0.9,and,0.571429,0.591301,0.176718,0.133929
2,split_c,0.425889,0.292308,0.094609,0.797101,0.898790,2.964586,0.9,and,0.558856,0.441026,0.215385,0.075908
3,split_d,0.224614,0.236671,0.091130,0.584171,0.940462,3.014245,0.9,and,0.535844,0.335501,0.165150,0.092345


,auroc_known_unknown,tpr_unknown,fpr_known,open_set_acc,known_accuracy,tau,kappa,stage1_pass_rate_known,stage1_pass_rate_unknown,tpr_unknown_system,fpr_known_system
mean,0.462624,0.206403,0.104843,0.558822,0.901187,2.962708,0.9,0.557357,0.551744,0.150941,0.094999


Wrote: /content/drive/MyDrive/fyp_outputs/severstral_osr/severstral_osr_summary_after_grid_best.csv
